In [1]:
import os
os.environ['BOKEH_ALLOW_WS_ORIGIN'] = ("1p28kn9d2v74i6ilhivhds70odo8uta4sugfufmvcsmi64u4r81g")
print("BOKEH_ALLOW_WS_ORIGIN =", os.environ.get('BOKEH_ALLOW_WS_ORIGIN'))


BOKEH_ALLOW_WS_ORIGIN = 1p28kn9d2v74i6ilhivhds70odo8uta4sugfufmvcsmi64u4r81g


In [2]:
!pip install matplotlib numpy scipy astropy
!pip install pyyaml bokeh mpl_point_clicker
!pip install lineid_plot pandas photutils

In [3]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
import sys

import matplotlib.pyplot as plt
import numpy as np

from astropy.io import fits

from bokeh.io import show, output_notebook
# # set local host for bokeh (change the number to the one in the url at the top of the page)
#localhost = 'localhost:8888'

output_notebook()

from photometryExercise import standardPlot



Loading BokehJS ...

In [30]:
# aperture photometry

direc = r'C:\Users\User\OneDrive - University of Edinburgh\Documents\TGP_Asteroids2_2025\Reduced_data_day1'

star_1_r_1 = fits.open(direc + r'\165659.fits')[0].data
# star_1_r_2 = fits.open(direc + r'\165696.fits')[0].data

standardPlot(star_1_r_1)

In [6]:
from photutils.aperture import aperture_photometry, CircularAperture, CircularAnnulus

In [24]:
xcor = 973
ycor = 986
rAperture = 5
rSkyInner = rAperture*1.5
rSkyOuter = rAperture*2

apertureStandard = CircularAperture((xcor, ycor), r=rAperture)
annulusSky = CircularAnnulus((xcor, ycor), r_in=rSkyInner, r_out=rSkyOuter)

In [25]:
# Let's make first an estimation of the sky values in your defined annuluss
sky = aperture_photometry(star_1_r_1, annulusSky)
print(sky)

 id xcenter ycenter    aperture_sum  
--- ------- ------- -----------------
  1   973.0   986.0 7145.081510646427


In [26]:
# To recover the mean sky value we should weight the total flux by the area of the annulus

area = np.pi * rSkyOuter**2 - (np.pi * rSkyInner**2)
skyFlux = sky['aperture_sum'].value # This is how we access astropy.table columns
meanSkyValue = skyFlux/area

print(f'The mean sky value is {meanSkyValue}')

The mean sky value is [51.98514474]


In [27]:
# Once we know the mean sky value, we can perform the aperture photometry on the standard star,
# removing first the sky value

standard = aperture_photometry(star_1_r_1 - meanSkyValue, apertureStandard)
standardFlux = standard['aperture_sum'].value
print(f'Total flux on aperture for the standard star {standardFlux}')

Total flux on aperture for the standard star [50060.59769019]


In [28]:
# Now, you should create a function that takes the total flux as input and returns a magnitude value
# Check the manual for details on this function

def flux_to_mag():
    return -2.5 * np.log10(standardFlux)

magnitude = flux_to_mag()
print(f'The magnitude of this standard star is {magnitude}')

The magnitude of this standard star is [-11.74874008]


In [31]:
def error_in_mag():
    t = skyFlux + standardFlux # maybe use meanSkyValue
    C = standardFlux
    dC = np.sqrt(standardFlux + skyFlux)

    return (-2.5 / np.log(10) ) * (dC / C) 

error_magnitude = error_in_mag()
print(error_magnitude)

[-0.00518738]


need to generalise above for all stars but cant find them rn
also idk if i can do it in a loop bc i think they might be in diff places

do i need to calculate instrumental mag idk what that is

now i have magnitude in each filter i can calculate colour for each